In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import lfilter

def pan_tompkins_preprocess(signal, fs=500):
    """
    Preprocesses an ECG signal using the Pan-Tompkins algorithm stages.
    Assumes a default sampling rate (fs) of 200 Hz. Adjust coefficients if fs differs significantly.
    """
    # 1. Low-pass Filter: y[n] = 2y[n-1] - y[n-2] + x[n] - 2x[n-6] + x[n-12]
    b_lp = np.zeros(13)
    b_lp[0], b_lp[6], b_lp[12] = 1, -2, 1
    a_lp = [1, -2, 1]
    lp_signal = lfilter(b_lp, a_lp, signal)
    
    # 2. High-pass Filter: y[n] = y[n-1] - x[n]/32 + x[n-16] - x[n-17] + x[n-32]/32
    b_hp = np.zeros(33)
    b_hp[0], b_hp[16], b_hp[17], b_hp[32] = -1/32, 1, -1, 1/32
    a_hp = [1, -1]
    hp_signal = lfilter(b_hp, a_hp, lp_signal)
    
    # 3. Derivative: y[n] = (1/8) * [2x[n] + x[n-1] - x[n-3] - 2x[n-4]]
    b_der = [2/8, 1/8, 0, -1/8, -2/8]
    der_signal = lfilter(b_der, 1, hp_signal)
    
    # 4. Squaring
    sq_signal = der_signal ** 2
    
    # 5. Moving Window Integration (Window size roughly 150ms -> 0.15 * fs)
    window_len = int(0.15 * fs)
    b_mwi = np.ones(window_len) / window_len
    mwi_signal = lfilter(b_mwi, 1, sq_signal)
    
    return mwi_signal

path = 'ecg_data.txt' 
original_ecg = np.loadtxt(path, delimiter='\t', usecols=1)

fs = 500 
preprocessed_ecg = pan_tompkins_preprocess(original_ecg, fs=fs)

# 3. Plotting the results
plt.figure(figsize=(12, 6))

# Original Signal Plot
plt.subplot(2, 1, 1)
plt.plot(original_ecg, color='b', label='Original ECG')
plt.title('Pan-Tompkins Preprocessing Pipeline')
plt.ylabel('Amplitude')
plt.grid(True)
plt.legend()

# Preprocessed Signal Plot
plt.subplot(2, 1, 2)
plt.plot(preprocessed_ecg, color='r', label='Preprocessed (MWI)')
plt.xlabel('Samples')
plt.ylabel('Amplitude')
plt.grid(True)
plt.legend()

plt.tight_layout()
plt.show()